# Stochastic Calculus, Step by Step · Lesson 20: The Black-Scholes formula

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. When is the call exercised? · 3. Two terms · 4. The strike term · 5. The share term: completing the square · 6. The formula, checked three ways · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 20,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. When is the call exercised?

Under the risk-neutral probability (Lessons 11 and 17), $S_T = S_0e^{(r - \frac12\sigma^2)T + \sigma\sqrt TZ}$ with $Z$ standard normal. The call pays when $S_T > K$, and taking logs turns that into a condition on $Z$:

$$Z > -d_-, \qquad d_- = \frac{\ln(S_0/K) + (r - \frac12\sigma^2)T}{\sigma\sqrt T} .$$

In [ ]:
dm, dp = float(d_minus(S0, T)), float(d_plus(S0, T))
out["ex"] = {"ln_ratio": math.log(S0 / K), "drift": (r - 0.5 * sigma**2) * T, "vol": sigma * math.sqrt(T), "d_minus": dm,
             "d_plus": dp, "z_star": -dm, "S_at_zstar": S0 * math.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * (-dm))}
out["ex"]

## 3. Two terms

The price is the discounted average payoff (Lesson 11). Split it where the payoff is not zero:

$$C = e^{-rT}\tilde E\big[(S_T - K)\mathbf 1\{S_T > K\}\big] = e^{-rT}\tilde E\big[S_T\mathbf 1\{S_T > K\}\big] - Ke^{-rT}\tilde P(S_T > K) .$$

In [ ]:
rng = np.random.default_rng(seed)
Z = rng.standard_normal(2_000_000)
ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Z)
disc = math.exp(-r * T)
out["split"] = {"share_mc": float(disc * (ST * (ST > K)).mean()), "strike_mc": float(K * disc * (ST > K).mean()),
                "p_mc": float((ST > K).mean()), "disc": disc, "K_disc": K * disc}
out["split"]

## 4. The strike term

$\tilde P(S_T > K) = P(Z > -d_-) = N(d_-)$, by the symmetry of the normal. So the second term is $Ke^{-rT}N(d_-)$.

In [ ]:
out["strike"] = {"N_minus": N(dm), "term": K * disc * N(dm)}
out["strike"]

## 5. The share term: completing the square

Write the first term as an integral against the normal density and complete the square (Lesson 4):

$$e^{\sigma\sqrt Tz}\varphi(z) = e^{\frac12\sigma^2T}\varphi\big(z - \sigma\sqrt T\big) .$$

The exponential tilts the bell curve into the same bell curve moved right by $\sigma\sqrt T$. The discount factors and drift then cancel, leaving $S_0$ times the area of the moved curve above $-d_-$, which is $N(d_-+ \sigma\sqrt T) = N(d_+)$.

In [ ]:
zg = np.linspace(-6, 7, 13_001)
za = np.linspace(-dm, 8, 8_001)   # from the critical z upward, so the grid starts exactly at the kink
tilted = np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * za) * phi(za) * disc * S0
integral = float(((tilted[1:] + tilted[:-1]) / 2 * np.diff(za)).sum())
out["share"] = {"N_plus": N(dp), "term": S0 * N(dp), "integral": integral, "shift": sigma * math.sqrt(T)}
zf = np.linspace(-4, 4.5, 341)
out["share"]["fig"] = {"z": zf.tolist(), "phi": phi(zf).tolist(), "phi_shift": phi(zf - sigma * math.sqrt(T)).tolist(), "z_star": -dm}
{k_: v for k_, v in out["share"].items() if k_ != "fig"}

## 6. The formula, checked three ways

$$C = S_0N(d_+) - Ke^{-rT}N(d_-), \qquad d_\pm = \frac{\ln(S_0/K) + (r \pm \frac12\sigma^2)T}{\sigma\sqrt T} .$$

Against a direct integral of payoff times density, against Monte Carlo with its error bar, and at other strikes:

In [ ]:
c_formula = S0 * N(dp) - K * disc * N(dm)
pay = disc * np.maximum(ST - K, 0)
payoff_z = np.maximum(S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * zg) - K, 0) * phi(zg) * disc
c_int = float(((payoff_z[1:] + payoff_z[:-1]) / 2 * np.diff(zg)).sum())
out["check"] = {"formula": c_formula, "integral": c_int, "mc": float(pay.mean()), "mc_se": float(pay.std() / math.sqrt(pay.size)),
                "put": K * disc * N(-dm) - S0 * N(-dp), "parity": c_formula - S0 + K * disc}
strike_rows = []
for k_ in (85.0, 95.0, 105.0, 115.0, 125.0):
    strike_rows.append({"K": k_, "d_plus": float(d_plus(S0, T, sigma, k_)), "d_minus": float(d_minus(S0, T, sigma, k_)),
                        "call": float(call(S0, T, sigma, k_)), "put": float(put(S0, T, sigma, k_)),
                        "mc": float(disc * np.maximum(ST - k_, 0).mean())})
out["check"]["rows"] = strike_rows
pd.DataFrame(strike_rows)

In [ ]:
# Chart data for the video (no random numbers)
mid_ = np.arange(-3.95, 4.0, 0.1)
out["share"]["hist"] = {"mid": mid_.tolist(), "plain": phi(mid_).tolist(), "shifted": phi(mid_ - sigma * math.sqrt(T)).tolist(),
                        "ticks": [{"x": float(x), "label": f"{x:.0f}"} for x in (-4, -2, 0, 2, 4)]}

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 4: Solutions*.

**Exercise 1.** Starting from $S_T = S_0e^{(r - \frac12\sigma^2)T + \sigma\sqrt TZ}$, show that $S_T > K$ exactly when $Z > -d_-$, and evaluate $d_-$ and the critical $Z$ with the course's numbers.

**Exercise 2.** Given $N(0.00605) = 0.50241$ and $N(-0.19395) = 0.42311$, compute the call price by hand, and say what each of the two terms is worth.

**Exercise 3.** Compute $e^{-rT}\tilde E[S_T\mathbf 1\{S_T > K\}]$ in full: write the integral, combine the exponents, complete the square, substitute, and show that it equals $S_0N(d_+)$.

**Exercise 4.** Derive the put formula $P = Ke^{-rT}N(-d_-) - S_0N(-d_+)$ by the same steps, and check that it agrees with put-call parity.

**Exercise 5.** A **cash digital** pays 1 if $S_T > K$; an **asset digital** pays $S_T$ if $S_T > K$. Write down their prices. Show that a call is an asset digital minus $K$ cash digitals, and evaluate all three.

**Exercise 6.** Find the limit of the call formula (a) as $\sigma \to 0$, (b) as $T \to 0$, (c) as $S_0 \to \infty$, and explain each in words.

**Exercise 7.** (Code) Price the call for strikes 85, 95, 105, 115, 125 by numerically integrating payoff times density (trapezium rule), and compare with the formula. How fine must the grid be for four decimals?

**Exercise 8.** (Code) Estimate the cash digital and the call by Monte Carlo with $10^6$ paths, with 95% confidence intervals. Which has the smaller error relative to its price, and why?

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")